## 03 - Camada Gold

Transforma as seis tabelas de `cine_medallion.silver` em um modelo dimensional em `cine_medallion.gold`, com cinco dimensões, uma tabela fato e três tabelas ponte.

- **Dimensões:** cadastro de filmes, gêneros, pessoas por função, produtoras e resumo de avaliações por filme.
- **Fato:** desempenho financeiro e engajamento, com uma linha por filme lançado até a data de execução.
- **Pontes:** vínculos entre filmes e gêneros, pessoas ou produtoras.

Execute as células em ordem no Databricks, após concluir a camada Silver. A sessão precisa de acesso às tabelas de origem e permissão para criar e gravar tabelas na Gold. A carga valida os dados antes de substituir as tabelas de destino; as chaves substitutas são recalculadas a cada execução.


In [0]:
# Configuração das tabelas de origem e destino.
# A data da sessão Spark será a referência única para selecionar filmes já lançados.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = "cine_medallion"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
data_execucao = spark.sql("SELECT current_date() AS data").first()["data"]

In [0]:
# Funções de validação de chaves e geração de identificadores para as dimensões.
# Interrompe a carga se houver chave nula ou combinação de chave duplicada.
def exigir_chave(df, colunas, tabela):
    for coluna in colunas:
        if df.filter(F.col(coluna).isNull()).limit(1).count():
            raise ValueError(f"{tabela}: chave nula em {coluna}")
    if df.groupBy(*colunas).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{tabela}: chave duplicada em {colunas}; corrigir a Silver")

# left_anti encontra referências sem correspondência na dimensão.
def exigir_fk(df, coluna, dimensao, tabela):
    if df.select(coluna).join(dimensao.select(coluna), coluna, "left_anti").limit(1).count():
        raise ValueError(f"{tabela}: referência órfã em {coluna}")

# Cria a chave substituta (sk) numerando as linhas pela ordenação das chaves naturais.
# Novos registros podem mudar essa numeração entre cargas; as tabelas relacionadas são reconstruídas.
def gerar_chave(df, naturais, substituta):
    return df.withColumn(substituta, F.row_number().over(
        Window.orderBy(*naturais)).cast("bigint"))

In [0]:
# Converte os tipos e seleciona somente as colunas definidas no contrato da tabela.
# Não transformar silenciosamente um valor preenchido em NULL na Gold.
def tipar(df, tipos, tabela):
    for coluna, tipo in tipos.items():
        convertido = F.expr(f"try_cast(`{coluna}` AS {tipo})")
        if df.filter(F.col(coluna).isNotNull() & convertido.isNull()).limit(1).count():
            raise ValueError(f"{tabela}: valor incompatível com {tipo} em {coluna}")
        # Impede que a conversão para inteiro descarte silenciosamente a parte decimal.
        if tipo == "int":
            numero = F.expr(f"try_cast(`{coluna}` AS DECIMAL(38,10))")
            if df.filter(numero.isNotNull() & (numero != F.floor(numero))).limit(1).count():
                raise ValueError(f"{tabela}: valor fracionário em {coluna}")
        df = df.withColumn(coluna, convertido)
    return df.select(*tipos)

In [0]:
# Repetição das funções auxiliares da célula anterior a tipar.
# Estas definições substituem as anteriores com a mesma lógica.
# Interrompe a carga se houver chave nula ou combinação de chave duplicada.
def exigir_chave(df, colunas, tabela):
    for coluna in colunas:
        if df.filter(F.col(coluna).isNull()).limit(1).count():
            raise ValueError(f"{tabela}: chave nula em {coluna}")
    if df.groupBy(*colunas).count().filter(F.col("count") > 1).limit(1).count():
        raise ValueError(f"{tabela}: chave duplicada em {colunas}; corrigir a Silver")

# left_anti encontra referências sem correspondência na dimensão.
def exigir_fk(df, coluna, dimensao, tabela):
    if df.select(coluna).join(dimensao.select(coluna), coluna, "left_anti").limit(1).count():
        raise ValueError(f"{tabela}: referência órfã em {coluna}")

# Cria a chave substituta (sk) numerando as linhas pela ordenação das chaves naturais.
# Novos registros podem mudar essa numeração entre cargas; as tabelas relacionadas são reconstruídas.
def gerar_chave(df, naturais, substituta):
    return df.withColumn(substituta, F.row_number().over(
        Window.orderBy(*naturais)).cast("bigint"))

In [0]:
# Mapeia os nomes usados no código para as seis tabelas de origem na Silver.
fontes = {
    "info": "tb_info_filmes",
    "financeiro": "tb_financeiro_filmes",
    "metricas": "tb_metricas_engajamento",
    "avaliacoes": "tb_avaliacoes_usuarios",
    "generos": "tb_generos",
    "entidades": "tb_pessoas_empresas",
}

In [0]:
# Lê as fontes e padroniza id_filme como string para os relacionamentos.
# info, financeiro e metricas devem ter uma única linha por id_filme presente.
entradas = {}
for nome, tabela in fontes.items():
    entradas[nome] = spark.table(f"{silver_schema}.{tabela}").withColumn(
        "id_filme", F.col("id_filme").cast("string")
    )
    # Dispara uma leitura da fonte; count não mantém o DataFrame em cache.
    entradas[nome].count()

for nome in ["info", "financeiro", "metricas"]:
    exigir_chave(entradas[nome], ["id_filme"], fontes[nome])


In [0]:
# Diagnóstico de registros cujo id_filme não existe no cadastro de filmes.
# O relatório conta linhas sem correspondência, não filmes distintos.
# Relacionamentos sem cadastro são exibidos e excluídos apenas na Gold.
ids = entradas["info"].select("id_filme")
orfandade = []
for nome in ["financeiro", "metricas", "avaliacoes", "generos", "entidades"]:
    quantidade = entradas[nome].join(ids, "id_filme", "left_anti").count()
    orfandade.append((fontes[nome], quantidade))
display(spark.createDataFrame(orfandade, "tabela STRING, registros_sem_filme BIGINT"))

# Tipos de entidade fora do domínio e nomes vazios interrompem a carga.
if entradas["entidades"].filter(
    F.col("tipo_entidade").isNull() |
    ~F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista", "Produtora")
).limit(1).count():
    raise ValueError("Tipos inválidos em tb_pessoas_empresas; corrigir a Silver")
for nome, coluna in [("generos", "nome_genero"), ("entidades", "nome_entidade")]:
    if entradas[nome].filter(F.col(coluna).isNull() | (F.trim(F.col(coluna)) == "")).limit(1).count():
        raise ValueError(f"{fontes[nome]}: nome ausente")

In [0]:
# Contrato da dimensão de filmes: nomes e tipos das colunas de negócio.
tipos_filmes = {
    "id_filme": "string", "titulo": "string", "data_lancamento": "date",
    "ano_lancamento": "int", "duracao_minutos": "int", "idioma_original": "string",
    "status_filme": "string", "sinopse": "string",
}

In [0]:
# Uma linha por filme cadastrado, inclusive títulos ainda não lançados.
# Mantém id_filme para rastrear a origem e adiciona sk_movie_id para os joins da Gold.
dim_movies = gerar_chave(tipar(entradas["info"], tipos_filmes, "dim_movies"),
                         ["id_filme"], "sk_movie_id").select("sk_movie_id", *tipos_filmes)


In [0]:
# Uma linha por nome de gênero distinto encontrado na Silver.
# A associação de cada gênero aos filmes será armazenada em uma tabela ponte.
dim_genres = gerar_chave(entradas["generos"].select("nome_genero").distinct(),
                         ["nome_genero"], "sk_genre_id").select("sk_genre_id", "nome_genero")

In [0]:
# Separa as pessoas das produtoras e usa nome + função como chave natural.
# O mesmo nome como Ator e Diretor gera duas linhas; homônimos na mesma função não são diferenciados.
pessoas = entradas["entidades"].filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")).select(
    "id_filme", F.col("nome_entidade").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa"))
dim_people = gerar_chave(pessoas.select("nome_pessoa", "tipo_pessoa").distinct(),
                         ["nome_pessoa", "tipo_pessoa"], "sk_person_id").select("sk_person_id", "nome_pessoa", "tipo_pessoa")

In [0]:
# Uma linha por nome de produtora distinto; os filmes associados ficam na ponte.
produtoras = entradas["entidades"].filter(F.col("tipo_entidade") == "Produtora").select(
    "id_filme", F.col("nome_entidade").alias("nome_produtora"))
dim_companies = gerar_chave(produtoras.select("nome_produtora").distinct(),
                            ["nome_produtora"], "sk_company_id").select("sk_company_id", "nome_produtora")

In [0]:
# Resume as avaliações em uma linha por filme com avaliações e cadastro válido.
# Remove apenas duplicatas completas das quatro colunas selecionadas, não por usuário.
avaliacoes = tipar(entradas["avaliacoes"], {
    "id_filme": "string", "nome_usuario": "string", "nota_usuario": "double", "comentario_usuario": "string"
}, "tb_avaliacoes_usuarios").dropDuplicates()
if avaliacoes.filter(F.col("nota_usuario").isNotNull() &
                    (~F.col("nota_usuario").between(0, 10) | F.isnan("nota_usuario"))).limit(1).count():
    raise ValueError("Nota de usuário fora da escala: corrigir a Silver")
# A contagem inclui notas nulas; a média considera apenas as notas preenchidas.
resumo_avaliacoes = avaliacoes.groupBy("id_filme").agg(
    F.count("*").alias("qtd_avaliacoes_usuarios"),
    F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios")
).join(dim_movies.select("id_filme", "sk_movie_id"), "id_filme", "inner")
resumo_avaliacoes = tipar(resumo_avaliacoes, {
    "id_filme": "string", "sk_movie_id": "bigint", "qtd_avaliacoes_usuarios": "int", "nota_media_usuarios": "double"
}, "dim_reviews")
dim_reviews = gerar_chave(resumo_avaliacoes, ["id_filme"], "sk_review_id").select(
    "sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")



In [0]:
# Prepara os valores financeiros e as métricas de engajamento para a fato.
# Os valores em USD e BRL já vêm da Silver; aqui são convertidos para decimal(18,2).
colunas_financeiras = ["orcamento_usd", "receita_usd", "lucro_usd", "orcamento_brl", "receita_brl", "lucro_brl"]
tipos_financeiros = {"id_filme": "string", **{c: "decimal(18,2)" for c in colunas_financeiras}}
tipos_metricas = {"id_filme": "string", "popularidade": "double", "nota_media_tmdb": "double",
                  "qtd_votos_tmdb": "int", "nota_media_imdb": "double", "qtd_votos_imdb": "int"}
financeiro = tipar(entradas["financeiro"], tipos_financeiros, "tb_financeiro_filmes")
metricas = tipar(entradas["metricas"], tipos_metricas, "tb_metricas_engajamento")
# Rejeita métricas negativas, NaN e infinitas; as notas também não podem superar 10.
# Valores nulos permanecem como informação ausente.
for coluna in ["popularidade", "qtd_votos_tmdb", "qtd_votos_imdb", "nota_media_tmdb", "nota_media_imdb"]:
    invalido = (F.col(coluna) < 0) | F.isnan(F.col(coluna).cast("double")) | (F.abs(F.col(coluna)) == float("inf"))
    if coluna.startswith("nota_"):
        invalido = invalido | (F.col(coluna) > 10)
    if metricas.filter(invalido).limit(1).count():
        raise ValueError(f"Métrica inválida na Silver: {coluna}")
# A fato inclui apenas status Lançado com data conhecida e não posterior à execução.
filmes_lancados = dim_movies.filter(
    (F.col("status_filme") == "Lançado") & F.col("data_lancamento").isNotNull() &
    (F.col("data_lancamento") <= F.lit(data_execucao)))
colunas_metricas = [c for c in tipos_metricas if c != "id_filme"]
# Os left joins mantêm uma linha por filme elegível, mesmo sem financeiro ou métricas.
# Nessas ausências, os valores ficam nulos, sem preenchimento artificial com zero.
fact_movies_performance = filmes_lancados.select("id_filme", "sk_movie_id").join(
    financeiro, "id_filme", "left").join(metricas, "id_filme", "left").select(
        "sk_movie_id", *colunas_financeiras, *colunas_metricas)

In [0]:
# Tabelas ponte representam relações muitos-para-muitos sem multiplicar as linhas da fato.
# Os inner joins retêm vínculos com filmes cadastrados; distinct remove pares repetidos.
# As pontes usam todos os filmes de dim_movies, sem o filtro de lançamento da fato.
mapa_filmes = dim_movies.select("id_filme", "sk_movie_id")
bridge_movie_genre = entradas["generos"].join(mapa_filmes, "id_filme", "inner").join(
    dim_genres, "nome_genero", "inner").select("sk_movie_id", "sk_genre_id").distinct()
bridge_movie_person = pessoas.join(mapa_filmes, "id_filme", "inner").join(
    dim_people, ["nome_pessoa", "tipo_pessoa"], "inner").select("sk_movie_id", "sk_person_id").distinct()
bridge_movie_company = produtoras.join(mapa_filmes, "id_filme", "inner").join(
    dim_companies, "nome_produtora", "inner").select("sk_movie_id", "sk_company_id").distinct()

### Contratos, validações e publicação

As próximas células definem as chaves e os tipos esperados, conferem a integridade dos relacionamentos e reconciliam os totais financeiros com a Silver para a mesma população de filmes. A gravação começa somente após essas verificações.


In [0]:
# Registro das nove tabelas a publicar e dos contratos usados na validação.
# Cada ponte tem chave composta: a combinação das duas chaves deve ser única.
tabelas_gold = {
    "dim_movies": dim_movies, "dim_genres": dim_genres, "dim_people": dim_people,
    "dim_companies": dim_companies, "dim_reviews": dim_reviews,
    "fact_movies_performance": fact_movies_performance,
    "bridge_movie_genre": bridge_movie_genre, "bridge_movie_person": bridge_movie_person,
    "bridge_movie_company": bridge_movie_company,
}
chaves = {
    "dim_movies": ["sk_movie_id"], "dim_genres": ["sk_genre_id"], "dim_people": ["sk_person_id"],
    "dim_companies": ["sk_company_id"], "dim_reviews": ["sk_review_id"],
    "fact_movies_performance": ["sk_movie_id"], "bridge_movie_genre": ["sk_movie_id", "sk_genre_id"],
    "bridge_movie_person": ["sk_movie_id", "sk_person_id"], "bridge_movie_company": ["sk_movie_id", "sk_company_id"],
}
esquemas_esperados = {
    "dim_movies": {"sk_movie_id": "bigint", **tipos_filmes},
    "dim_genres": {"sk_genre_id": "bigint", "nome_genero": "string"},
    "dim_people": {"sk_person_id": "bigint", "nome_pessoa": "string", "tipo_pessoa": "string"},
    "dim_companies": {"sk_company_id": "bigint", "nome_produtora": "string"},
    "dim_reviews": {"sk_review_id": "bigint", "sk_movie_id": "bigint", "qtd_avaliacoes_usuarios": "int", "nota_media_usuarios": "double"},
    "fact_movies_performance": {"sk_movie_id": "bigint", **{c: "decimal(18,2)" for c in colunas_financeiras},
                                **{c: tipos_metricas[c] for c in colunas_metricas}},
    "bridge_movie_genre": {"sk_movie_id": "bigint", "sk_genre_id": "bigint"},
    "bridge_movie_person": {"sk_movie_id": "bigint", "sk_person_id": "bigint"},
    "bridge_movie_company": {"sk_movie_id": "bigint", "sk_company_id": "bigint"},
}

In [0]:
# Valida todas as tabelas antes de iniciar a gravação: chaves, tipos e referências.
for nome, df in tabelas_gold.items():
    df
    exigir_chave(df, chaves[nome], nome)
    if dict(df.dtypes) != esquemas_esperados[nome]:
        raise ValueError(f"Schema incorreto em {nome}: {df.dtypes}")
# Além da chave substituta, garante no máximo um resumo de avaliações por filme.
exigir_chave(dim_reviews, ["sk_movie_id"], "dim_reviews")
for nome in ["dim_reviews", "fact_movies_performance", "bridge_movie_genre", "bridge_movie_person", "bridge_movie_company"]:
    exigir_fk(tabelas_gold[nome], "sk_movie_id", dim_movies, nome)
for nome, chave, dimensao in [
    ("bridge_movie_genre", "sk_genre_id", dim_genres),
    ("bridge_movie_person", "sk_person_id", dim_people),
    ("bridge_movie_company", "sk_company_id", dim_companies),
]:
    exigir_fk(tabelas_gold[nome], chave, dimensao, nome)
# Confere que os joins preservaram a quantidade de filmes elegíveis na fato.
if fact_movies_performance.count() != filmes_lancados.count():
    raise ValueError("Os joins alteraram o grão da fato")

# Reconciliação: mesma população e mesmas seis métricas da Silver.
origem_elegivel = financeiro.join(filmes_lancados.select("id_filme"), "id_filme", "inner")
somas = [F.sum(c).alias(c) for c in colunas_financeiras]
if origem_elegivel.agg(*somas).first() != fact_movies_performance.agg(*somas).first():
    raise ValueError("Totais financeiros da fato divergentes da Silver")
print("Validações de schema, chaves, grão, referências e totais concluídas.")

In [0]:
# Publica a carga completa em Delta, substituindo dados e schema de cada tabela.
# A gravação é feita tabela a tabela: não há uma transação única para toda a Gold.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")
for nome, df in tabelas_gold.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.{nome}")
    print(f"Gravada: {gold_schema}.{nome}")

# Exibe as contagens dos DataFrames publicados e uma amostra da fato para inspeção.
display(spark.createDataFrame(
    [(nome, df.count()) for nome, df in tabelas_gold.items()], "tabela STRING, quantidade_linhas BIGINT"))
display(fact_movies_performance.orderBy("sk_movie_id").limit(10))